CREARE PIPELINE DI INFERENZA LOCALE (TRANSFORMERS + FASTAPI)

Cosa succede quando dobbiamo essere noi i padroni di casa, quando il medello deve girare sui nostri server?

- Model Caching: strategie di caricamente all'avvio del server, tenere il modello sempre pronto all'uso
- Batching Dinamico: tecnica segreta per far lavorare la GPU al massimo, processando più richieste insieme
- Threadpool e Concorrenza: come evitare con un inferenza pesante congeli il nostro server, rendendolo indisponibile per tutti gli altri utenti.

Perchè caricare il modello è un'operazione così critia?

Model Caching e Ciclo di Vita
Evitare il caricamento ridondante dei pesi.
Caricare un modello da miliardi di parametri ad ogni richiesta HTTP porterebbe a latenze inaccettabili e saturazione del bus di memoria, perchè sposta dati dal disco alla memoria video.
E' quindi un'operazione lenta e se dovessimo fare questo caricamente dentro la funzione che risponde all'utente la latenza diventerebbe inacettabile.
Il model caching serve proprio a questo, carichiamo i pesi una sola volta all'avvio del server, rendendo l'AI residente e immediatamente disponibile.
Utilizzando gli eventi di startup di FastAPI o il contesto 'lifespan' per garantire che i pesi siano pronti in VRAM prima di accettare la prima connessione.

Vediamo come Fast API ci permette di gestire questo ciclo di vita in modo elegante.

Residenzialità in Memoria
Persistenza dei pesi e Singleton
- Startup Events: funzioni decorate con @app.on_event("startup") per inizilizzare i modelli prima dell'apertura delle porte di rete. Per dire a FastAPI: "non aprire le porte al pubblico finchè il modello non è pronto".
- Singleton Pattern: utilizzo di variabili globali o dipendenze iniettate per garantire che esista una sola istanza del modello in memoria, però accessibile da tutte le richieste.
- VRAM Management: monitoraggio dello spazio occupato per evitare 'out of memory' durante l'allocazione iniziale dei tensori. Se carichiamo un modello troppo grande senza controllare lo spazio, il sistema potrebbe fallire con un errore out of memory
- Il tempo di caricamento iniziale è trascurabile se comparato al risparmio di latenza su migliaia di inferenze successive.

Il tempo di inferenza deve essere l'unica cosa che l'utente percepisce, il tempo di caricamento deve essere un costo pagato una sola volta all avvio

Ma come scriviamo questo all'interno del nostro codice Python?

Implementazione Tecnica
- Lifespan Context Manager: la moderna alternativa agli eventi di startup che permette di gestire in modo pulito sia l'allocazione che il rilascio delle risorse GPU. E' come un rito di apertura e chisura del server, allochiamo e liberiamo le risorse in modo pulito.
- Lazy Loading vs Eager Loading: valutazione del caricamento al primo utilizzo rispetto al caricamento all'avvio per bilanciare i tempi di boot del container.
- Warm-up Inference: esecuzione di una predizione fittizia all'avvio (con dati casuali) per inizializzare i grafi computazionali di CUDA e stabilizzare la latenza. Controlliamo che il modello funzioni alla partenza, perchè le librerie come CUDA spesso ottimizzano il calcolo solo dopo il primo passaggio.
In questo modo, il primo utente reale, non subisce un ritardo tecnico dovuto all'inizializzazione dei grafi.

Ma questo investimento iniziale di tempo, quanto ci ripaga intermi di prestazioni?

Analisi del Throuthput
Ottimizzazione del Boot
Il caching trasforma l'applicazione da un sistema 'cold-start' a un servizio 'hot-ready', riducendo drasticamente la varianza dei tempi di risposta.
Il costo computazionale del primo caricamento viene ammortizzato linearmente con l'aumentare delle richieste processate.

Ora che il modello è in memoria, come possiamo sfruttare ogni singolo ciclo della nostra GPU?

Batching Dinamico in Produzione
Massimizzare il parallelismo hardware
Processare una richiesta alla volta è inefficiente per le moderne GPU, che sono progettate per operazioni SIMD sularga scala.
Il batching dinamico raggruppa le richieste che arrivano in un piccolo intervallo temporale per processarle in un unico passaggio matriciale.
Immagina un autobus, è più efficiente apsettare che 10 persona salgano a bordo invece di fare 10 viaggi per ogni passeggero
Se alla GPU dai una sola frase alla volta, utilizzi la GPU in modo parziale.

Questo ci porta ad una sfida di bilanciamento: quanto tempo è giusto far aspettare il primo passeggero?

Saturazione della GPU
Bilanciare latenza e volume
- Request Queuing: accumulo delle richieste in arrivo in una coda thread-safe per un tempo massimo di pochi millisecondi. Invece di eseguire l'inferenza appena arriva il dato, la mettiamo in una coda. Se la coda si riempie o se scade un timer, inviamo tutto il batch alla GPU
- Parallel Inference: esecuzione del modello su un tensore di dimensione (B,L,D) dove B è il numero di richieste del batch
- Wait Time Trade-off: impostazione del tempo massimo di attesa per formare il batch per non penalizzare l'utente singolo
- In throughput del sistema aumenta fino a raggiungere il limite fisico di memoria della GPU

Ma gestire batch di dati che provengono da utenti diversi, richiede accortezze tecniche.

Tecniche di Implementazione
- Padding Adattivo: gestione delle diverse lunghezze degli input all'interno dello stesso batch per minimizzare i calcoli sui token di riempimento. Le richieste degli utenti non sono mai identiche, uno scrive un paragrafo, uno una sola parola. Per mettere insieme tutto questo in una matrice dobbiamo usare il padding adattivo, aggiungendo degli  più lulghe.si più corte per pareggiarle a quelle più lunghe.
- Adaptive Batching: algoritmi che modificano la dimensione del batch in tempo reale basandosi sul monitoraggio del carico istantaneo.
- Inference Engines: utilizzo di librerie esterne come BentoML o Ray Serve che implementano il batching dinamico in modo trasparente.

Il batching dinamico è ciò che permette ad un server di gestire 1 utente a 100 utenti contemporaneamente.

Il guadagno prestazionale deriva dalla riduzione del numero di trasferimento di dati tra CPU e GPU e dalla saturazione dei Tensor Cores.
Esiste un limite oltre il quale l'aumento del batch size non produce più benefici a causa del collo di bottiglia sulla banda di memoria.
Finchè non raggiungiamo questo limite fisico, raggruppare è la scelta vincente.

Ora dobbiaom assicurarci che il nostro server rimanga sveglio

Threadpool e Gestione Concorrente
Evitare il blocco dell'Event Loop
FastAPI è asincrono, ma l'inferenza del Deep Learning è un task puramente CPU/GPU-bound che blocca il thread corrente per tutta la sua durate.
E' possibile delegare queste operazioni pesanti a thread separati per permettere al serveer di continuare a gestire altre connessioni I/O

Architetture Professionali
- Worker Processes: configurazione di Uvicorn con più worker per bypassare il GIL e sfruttare core CPU multi per il preprocessing
- Task Queues: spostamento dell'inferenza pesante su code asincrone come Celery o Redis Queue per risposte disaccopiate
- Global Interpreter Lock (GIL): richiamo alla necessità di processi separati quando l'elaborazione avviene prevalentemente su CPU anzichè delegata alla GPU

Reattività del Server
Metriche di risposta
Un server correttamente configurato deve rispondere ai ping di 'health check' anche mentre la GPU è impegnata al 100%
L'isolamento dell'inferenza in thredpool garantisce che la latenza di rete non venga influenzata dal carico computazionale.

In [ ]:
import asyncio
import time
import torch
import torch.nn as nn
import torch.optim as optim
from fastapi import FastAPI
from contextlib import asynccontextmanager
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- SEZIONE 1: CONFIGURAZIONE E STRUTTURE DATI ---
# Questo intervallo definisce quanto tempo il server aspetta per accumulare richieste
# prima di processarle. Un valore di 1.0s e' molto alto per scopi dimostrativi,
# per permetterci di inviare piu' richieste manualmente e vederle gestite insieme.
BATCH_INTERVAL = 1.0 

# model_storage funge da memoria condivisa globale per mantenere il modello e gli scaler
# accessibili in tutta l'applicazione (pattern simile a un Singleton).
model_storage = {}

# request_queue e' il canale di comunicazione tra l'endpoint API (produttore)
# e il loop di processamento batch (consumatore). E' thread-safe e async-safe.
request_queue = asyncio.Queue()

# --- SEZIONE 2: DEFINIZIONE DEL MODELLO NEURALE ---
# Definiamo una semplice rete neurale con PyTorch per classificare i fiori Iris.
# Questa classe estende nn.Module, la classe base per tutti i moduli di reti neurali in PyTorch.
class SimpleIrisNet(nn.Module):
    def __init__(self):
        super(SimpleIrisNet, self).__init__()
        # Primo livello lineare: prende 4 feature in input e produce 16 output nascosti
        self.fc1 = nn.Linear(4, 16)
        # Secondo livello lineare: prende i 16 output nascosti e produce 3 output finali (le 3 classi)
        self.fc2 = nn.Linear(16, 3) 
        # Funzione di attivazione ReLU per introdurre non linearita'
        self.relu = nn.ReLU()
        # Softmax per convertire gli output in probabilita' (somma = 1)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # Il metodo forward definisce come i dati attraversano la rete
        x = self.relu(self.fc1(x))
        x = self.softmax(self.fc2(x))
        return x

# --- SEZIONE 3: IL MOTORE DI BATCHING (CONSUMER) ---
# Questa funzione asincrona gira in background per tutta la vita dell'applicazione.
# Il suo compito e' prelevare richieste dalla coda, raggrupparle ed eseguire l'inferenza.
async def batch_processing_loop():
    print(f"Batch processor avviato. Accumulo richieste per {BATCH_INTERVAL}s...")
    
    while True:
        # FASE A: Attesa (Windowing)
        # Il processore dorme per l'intervallo definito. Questo crea la "finestra temporale"
        # in cui le richieste in arrivo si accumulano nella coda.
        await asyncio.sleep(BATCH_INTERVAL)
        
        # FASE B: Raccolta (Fetching)
        batch_items = []
        # Svuotiamo la coda di tutte le richieste arrivate durante l'attesa.
        # Utilizziamo get_nowait() perche' non vogliamo bloccarci se la coda e' vuota.
        while not request_queue.empty():
            try:
                item = request_queue.get_nowait()
                batch_items.append(item)
            except asyncio.QueueEmpty:
                break
            
        # Se non ci sono richieste, torniamo all'inizio del loop (attesa)
        if not batch_items:
            continue 
            
        print(f"Processando batch di {len(batch_items)} richieste...")
        
        # FASE C: Preparazione del Batch
        # Separiamo i dati di input (features) dai futuri (le "promesse" di risposta).
        inputs = [item['features'] for item in batch_items]
        futures = [item['future'] for item in batch_items]
        
        # FASE D: Inferenza Batch (Il cuore dell'ottimizzazione)
        # Invece di passare un singolo esempio alla volta, creiamo un tensore che contiene
        # TUTTI gli esempi. Questo permette a PyTorch di parallelizzare i calcoli matriciali.
        # E' qui che otteniamo il guadagno di performance rispetto all'elaborazione sequenziale.
        inputs_tensor = torch.tensor(inputs, dtype=torch.float32)
        
        with torch.no_grad(): # Disabilitiamo il calcolo dei gradienti per risparmiare memoria
            outputs = model_storage['model'](inputs_tensor)
            # Otteniamo l'indice della classe con la probabilita' piu' alta per ogni esempio
            _, predicted_indices = torch.max(outputs, 1)
            
        # FASE E: Risoluzione delle Richieste
        results = predicted_indices.tolist()
        class_names = model_storage['class_names']
        
        # Ora dobbiamo restituire il risultato corretto a ogni richiesta in attesa.
        # Iteriamo su coppie (future, risultato) e impostiamo il risultato nel future.
        # Questo sblocchera' l'endpoint API che sta aspettando quella specifica risposta.
        for future, idx in zip(futures, results):
            if not future.done():
                future.set_result(class_names[idx])


# --- SEZIONE 4: GESTIONE DEL CICLO DI VITA (LIFESPAN) ---
# Questa funzione gestisce le operazioni di avvio e spegnimento dell'applicazione.
@asynccontextmanager
async def lifespan(app: FastAPI):
    # --- AVVIO (STARTUP) ---
    print("Caricamento e training modello Iris...")
    
    # 1. Caricamento Dati
    iris = load_iris()
    X, y = iris.data, iris.target
    model_storage['class_names'] = iris.target_names
    
    # 2. Preprocessing (Standardizzazione)
    # Le reti neurali lavorano meglio con dati scalati (media 0, varianza 1).
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    model_storage['scaler'] = scaler 
    
    # 3. Addestramento "al volo"
    # Creiamo istanza del modello, funzione di perdita e ottimizzatore
    model = SimpleIrisNet()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.05)
    
    # Convertiamo i dati numpy in tensori PyTorch
    X_t = torch.tensor(X_scaled, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    
    # Loop di addestramento (150 epoche)
    for _ in range(150): 
        optimizer.zero_grad() # Reset dei gradienti
        output = model(X_t)   # Forward pass
        loss = criterion(output, y_t) # Calcolo errore
        loss.backward()       # Backpropagation
        optimizer.step()      # Aggiornamento pesi
        
    model_storage['model'] = model
    print("Modello addestrato e pronto.")
    
    # 4. Avvio Task in Background
    # Avviamo il loop infinito del batch processor come task asyncio indipendente.
    batch_task = asyncio.create_task(batch_processing_loop())
    
    yield # Punto di sospensione: qui il server inizia a servire le richieste HTTP
    
    # --- SPEGNIMENTO (SHUTDOWN) ---
    # Quando il server si ferma, cancelliamo il task di background per chiudere pulito.
    batch_task.cancel()
    print("Server spento.")

# Inizializzazione dell'app FastAPI con il gestore lifespan
app = FastAPI(lifespan=lifespan)

# --- SEZIONE 5: ENDPOINT API (PRODUCER) ---
@app.post("/predict")
async def predict(features: list[float]):
    """
    Endpoint per la predizione della classe del fiore Iris.
    Non esegue la predizione immediatamente, ma delega al batch processor.
    """
    start_time = time.time()
    
    # 1. Validazione Input
    if len(features) != 4:
        return {"error": "Servono esattamente 4 features (sepal length, sepal width, petal length, petal width)"}
        
    # 2. Preprocessing Speculare al Training
    # Dobbiamo scalare i dati di input esattamente come abbiamo scalato i dati di training.
    scaler = model_storage['scaler']
    features_scaled = scaler.transform([features])[0]
    
    # 3. Creazione del Future
    # Otteniamo il loop di eventi corrente e creiamo un oggetto Future.
    # Un Future e' un oggetto che rappresentera' un risultato che non esiste ancora.
    loop = asyncio.get_running_loop()
    future = loop.create_future() 
    
    # 4. Inserimento in Coda
    # Inseriamo nella coda un dizionario contenente i dati da elaborare e il future associato.
    # Il batch processor trovera' questo pacchetto e lo elaborera'.
    await request_queue.put({
        'features': features_scaled,
        'future': future
    })
    
    # 5. Attesa Asincrona
    # Qui avviene la magia: await future sospende l'esecuzione di QUESTA specifica richiesta.
    # FastAPI e il loop asyncio sono liberi di gestire altre richieste mentre questa aspetta.
    # L'esecuzione riprendera' solo quando il batch processor chiamera' future.set_result().
    prediction = await future
    
    # 6. Risposta al Client
    elapsed = time.time() - start_time
    return {
        "class": prediction,
        "latency": f"{elapsed:.2f}s",
        "mode": "Batched (1s window)"
    }

# Nota per l'esecuzione:
# Eseguire questo file con: uvicorn 'esempio_60:app' --reload

: 